# 1. Import thư viện và các Module cần thiết

In [1]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import pandas as pd

# Định nghĩa đường dẫn gốc dự án (PROJECT_DIR)
PROJECT_DIR = Path.cwd().parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

# Import Data Pipeline 
from src.data import DataConfig, build_dataloaders

# Import Training Engine và TensorBoard Utils
from src.trainer import (
    train_one_epoch, 
    validate_one_epoch, 
    train_model, 
    save_checkpoint, 
    get_device
)
from src.tensorboard_utils import TensorBoardLogger

print(f" Đường dẫn dự án (PROJECT_DIR): {PROJECT_DIR}")

 Đường dẫn dự án (PROJECT_DIR): q:\Deep_Learning\UTH-Deep-Learning-nhom2\Practice_2


# 2. Nhận DataLoader từ src/data.py (Không chia lại data, dùng seed=42)

In [2]:
# Cấu hình data_dir = PROJECT_DIR / "data"
config = DataConfig(
    data_dir=PROJECT_DIR / "data",
    image_size=224,
    batch_size=32,
    val_ratio=0.10, # Giữ nguyên tỷ lệ 10% validation
    num_workers=2,
    seed=42          # Giữ nguyên seed=42 để dùng chung dữ liệu với cả nhóm
)

# Gọi hàm build_dataloaders dùng chung từ src/data.py
data_dict = build_dataloaders(config)

train_loader = data_dict['train_loader']
val_loader = data_dict['val_loader']
test_loader = data_dict['test_loader']
class_names = data_dict['class_names']

print(" DataLoaders của Tấn Lên đã sẵn sàng!")
print(f"- Số lượng batch Train: {len(train_loader)}")
print(f"- Số lượng batch Val:   {len(val_loader)}")
print(f"- Số lượng batch Test:  {len(test_loader)}")
print(f"- Danh sách 10 lớp: {class_names}")

100%|██████████| 170M/170M [19:46<00:00, 144kB/s]  


 DataLoaders của Tấn Lên đã sẵn sàng!
- Số lượng batch Train: 1407
- Số lượng batch Val:   157
- Số lượng batch Test:  313
- Danh sách 10 lớp: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


# 3. Xác định device CPU/GPU

In [3]:
device = get_device()
print(f"Thiết bị tính toán được tự động chọn: {device}")
if device.type == 'cuda':
    print(f" GPU Name: {torch.cuda.get_device_name(0)}")

Thiết bị tính toán được tự động chọn: cpu


# 4. Chuẩn bị model test (ResNet-18)

In [4]:
def create_resnet18_model(num_classes=10):
    """Tải ResNet-18 pre-trained và thay thế lớp classifier 10 đầu ra."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

model_test = create_resnet18_model()
print(" Mô hình ResNet-18 đã được cấu hình sẵn sàng thử nghiệm!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\QUY/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 24.9MB/s]


 Mô hình ResNet-18 đã được cấu hình sẵn sàng thử nghiệm!


# 5. Loss Function 

In [5]:
criterion = nn.CrossEntropyLoss()
print(" Hàm mất mát (Loss Function): CrossEntropyLoss")

 Hàm mất mát (Loss Function): CrossEntropyLoss


# 6. Optimizer (Khởi tạo các tham số huấn luyện)

In [6]:
# Thiết lập các siêu tham số chung cho thí nghiệm so sánh
LR = 0.001
NUM_EPOCHS = 5 # Số epoch chạy thử nghiệm so sánh Optimizer

# 7. Kiểm tra hàm

In [7]:
print("--- Kiểm tra chạy thử 1 Epoch Huấn luyện ---")
demo_model = create_resnet18_model().to(device)
demo_optimizer = optim.Adam(demo_model.parameters(), lr=LR)

# Gọi trực tiếp train_one_epoch()
train_loss_demo, train_acc_demo = train_one_epoch(
    model=demo_model, 
    dataloader=train_loader, 
    criterion=criterion, 
    optimizer=demo_optimizer, 
    device=device
)

print(f"Результат Train 1 Epoch -> Loss: {train_loss_demo:.4f} | Accuracy: {train_acc_demo:.2f}%")

--- Kiểm tra chạy thử 1 Epoch Huấn luyện ---


Training: 100%|██████████| 1407/1407 [38:48<00:00,  1.65s/it]

Результат Train 1 Epoch -> Loss: 0.7547 | Accuracy: 74.22%


# 8. Kiểm tra hàm validate_one_epoch()

In [8]:
print("--- Kiểm tra chạy thử 1 Epoch Validation ---")

# Gọi trực tiếp validate_one_epoch()
val_loss_demo, val_acc_demo = validate_one_epoch(
    model=demo_model, 
    dataloader=val_loader, 
    criterion=criterion, 
    device=device
)

print(f"Результат Validation 1 Epoch -> Loss: {val_loss_demo:.4f} | Accuracy: {val_acc_demo:.2f}%")

--- Kiểm tra chạy thử 1 Epoch Validation ---


Validating: 100%|██████████| 157/157 [01:38<00:00,  1.59it/s]

Результат Validation 1 Epoch -> Loss: 0.4894 | Accuracy: 82.22%


# 9, 10, 11, 12, 13. Tích hợp train_model(), Checkpoint & TensorBoard Logging

In [9]:
# Đảm bảo các thư mục checkpoints và runs được khởi tạo đầy đủ
(PROJECT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "runs").mkdir(parents=True, exist_ok=True)

print(" Khung hàm train_model() đã tích hợp sẵn:")
print("  - [10] Tự động lưu Best Checkpoint dựa trên Validation Accuracy")
print("  - [11] Khởi tạo TensorBoard Writer")
print("  - [12] Ghi log Train Loss & Validation Loss")
print("  - [13] Ghi log Train Accuracy & Validation Accuracy")

 Khung hàm train_model() đã tích hợp sẵn:
  - [10] Tự động lưu Best Checkpoint dựa trên Validation Accuracy
  - [11] Khởi tạo TensorBoard Writer
  - [12] Ghi log Train Loss & Validation Loss
  - [13] Ghi log Train Accuracy & Validation Accuracy


# 14.1 So sánh Adam và SGD — Thí nghiệm A: Adam

In [10]:
print("="*60)
print(" THÍ NGHIỆM A: ResNet-18 + Optimizer ADAM")
print("="*60)

model_adam = create_resnet18_model()
optimizer_adam = optim.Adam(model_adam.parameters(), lr=LR)

log_dir_adam = str(PROJECT_DIR / "runs" / "resnet18_adam")
checkpoint_adam = str(PROJECT_DIR / "checkpoints" / "best_resnet18_adam.pth")

history_adam = train_model(
    model=model_adam,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_adam,
    num_epochs=NUM_EPOCHS,
    log_dir=log_dir_adam,
    checkpoint_path=checkpoint_adam
)

 THÍ NGHIỆM A: ResNet-18 + Optimizer ADAM

Epoch [1/5]


Validating: 100%|██████████| 157/157 [01:30<00:00,  1.74it/s]


Train Loss: 0.7459, Train Acc: 74.64%
Val Loss: 0.6025, Val Acc: 80.52%
Validation accuracy improved (0.00% --> 80.52%). Saving model...

Epoch [2/5]


Validating: 100%|██████████| 157/157 [01:26<00:00,  1.83it/s]


Train Loss: 0.4706, Train Acc: 83.99%
Val Loss: 0.4033, Val Acc: 86.66%
Validation accuracy improved (80.52% --> 86.66%). Saving model...

Epoch [3/5]


...

Train Loss: 0.3788, Train Acc: 87.23%
Val Loss: 0.3413, Val Acc: 88.38%
Validation accuracy improved (86.66% --> 88.38%). Saving model...

Epoch [4/5]


Validating: 100%|██████████| 157/157 [01:25<00:00,  1.84it/s]


Train Loss: 0.3216, Train Acc: 89.10%
Val Loss: 0.3563, Val Acc: 87.98%

Epoch [5/5]


Validating: 100%|██████████| 157/157 [01:23<00:00,  1.89it/s]


Train Loss: 0.2798, Train Acc: 90.38%
Val Loss: 0.2784, Val Acc: 90.86%
Validation accuracy improved (88.38% --> 90.86%). Saving model...


# 14.2 So sánh Adam và SGD — Thí nghiệm B: SGD với Momentum

In [11]:
print("="*60)
print(" THÍ NGHIỆM B: ResNet-18 + Optimizer SGD (Momentum = 0.9)")
print("="*60)

model_sgd = create_resnet18_model()
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=LR, momentum=0.9)

log_dir_sgd = str(PROJECT_DIR / "runs" / "resnet18_sgd")
checkpoint_sgd = str(PROJECT_DIR / "checkpoints" / "best_resnet18_sgd.pth")

history_sgd = train_model(
    model=model_sgd,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_sgd,
    num_epochs=NUM_EPOCHS,
    log_dir=log_dir_sgd,
    checkpoint_path=checkpoint_sgd
)

 THÍ NGHIỆM B: ResNet-18 + Optimizer SGD (Momentum = 0.9)

Epoch [1/5]


Validating: 100%|██████████| 157/157 [01:22<00:00,  1.91it/s]


Train Loss: 0.4429, Train Acc: 85.42%
Val Loss: 0.2034, Val Acc: 92.76%
Validation accuracy improved (0.00% --> 92.76%). Saving model...

Epoch [2/5]


Validating: 100%|██████████| 157/157 [01:25<00:00,  1.83it/s]


Train Loss: 0.2108, Train Acc: 92.85%
Val Loss: 0.1610, Val Acc: 94.46%
Validation accuracy improved (92.76% --> 94.46%). Saving model...

Epoch [3/5]


...

Train Loss: 0.1510, Train Acc: 94.90%
Val Loss: 0.1490, Val Acc: 94.78%
Validation accuracy improved (94.46% --> 94.78%). Saving model...

Epoch [4/5]


Validating: 100%|██████████| 157/157 [01:29<00:00,  1.76it/s]


Train Loss: 0.1218, Train Acc: 95.99%
Val Loss: 0.1329, Val Acc: 95.42%
Validation accuracy improved (94.78% --> 95.42%). Saving model...

Epoch [5/5]


Validating: 100%|██████████| 157/157 [01:21<00:00,  1.92it/s]


Train Loss: 0.0983, Train Acc: 96.71%
Val Loss: 0.1303, Val Acc: 95.66%
Validation accuracy improved (95.42% --> 95.66%). Saving model...


# 15. Tổng hợp Bảng so sánh kết quả

In [12]:
results_data = [
    {
        "Optimizer": "Adam",
        "Learning Rate": LR,
        "Best Val Acc (%)": max(history_adam['val_acc']),
        "Min Val Loss": min(history_adam['val_loss']),
        "Final Train Acc (%)": history_adam['train_acc'][-1],
        "Final Val Acc (%)": history_adam['val_acc'][-1]
    },
    {
        "Optimizer": "SGD + Momentum (0.9)",
        "Learning Rate": LR,
        "Best Val Acc (%)": max(history_sgd['val_acc']),
        "Min Val Loss": min(history_sgd['val_loss']),
        "Final Train Acc (%)": history_sgd['train_acc'][-1],
        "Final Val Acc (%)": history_sgd['val_acc'][-1]
    }
]

df_compare = pd.DataFrame(results_data)
print("\n BẢNG SO SÁNH KẾT QUẢ TỐI ƯU HÓA:")
display(df_compare)


 BẢNG SO SÁNH KẾT QUẢ TỐI ƯU HÓA:


,Optimizer,Learning Rate,Best Val Acc (%),Min Val Loss,Final Train Acc (%),Final Val Acc (%)
0,Adam,0.001,90.86,0.278430,90.382222,90.86
1,SGD + Momentum (0.9),0.001,95.66,0.130289,96.706667,95.66
